# 第4回 線形回帰の評価プロトコルを実装し、結果を解釈する


---


## 前回からの接続


第3回は、民泊のホストとして、データを整えてから線形回帰（linear regression）の式を作り、1泊の値段を付けました。整えるたびに、テストでの平均絶対誤差（MAE）を測って比べました。

ただ、そのMAEは、訓練とテストへの分け方1通りで測った数でした。分け方を変えたら、数はどれくらい変わるのでしょうか。そして、式の係数が何を言っているのかは、まだ読んでいません。今日は、住宅の価格のデータで、式の評価と係数の解釈に取り組みます。


---


## 今日の分析目標


あなたは住宅の査定担当者です。お客さんに説明できる査定の式を、どう選ぶかに答える。

米国エイムズ市の住宅の売買2,412軒のデータで、価格を当てる線形回帰の式を作ります。式は、当たりの良さと、係数がお客さんへの説明に使えるかの2つの面から選び、1軒の家の査定書を書きます。

もう1つのねらいは、自分で書いた関数を部品にして、大きな関数を組み立てられるようになることです。誤差を測る関数を、分け方を何度も変えて評価する関数の中で呼び、その関数を列の組ごとに呼んで表に並べます。最後の「目標に答えられたか」で振り返りましょう。


### 場面

あなたは、米国アイオワ州エイムズ市の住宅の査定担当者です。家の条件（広さ・品質・建てた年など）から値段を見積もり、お客さんに伝えます。

お客さんは、値段だけでは納得しません。「なぜこの値段なのか」を、条件ごとに説明する必要があります。


### 手元のデータ

エイムズ市の資産評価事務所が記録した、2006〜2010年の住宅の売買2,930件です（De Cock, 2011）。1行が1軒の売買で、列は80本あります。答えの列は、その家が売れた価格（米ドル）です。

このうち、ふつうの売買で、広すぎない家に絞って使います（0節）。


### 最後に出す答え

査定に使う式と、それを選んだ理由です。そして、その式で書いた1軒の家の査定書です。査定書には、査定額・どれくらい外れうるか・平均的な家より高い（安い）理由を、条件ごとの金額で書きます。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- 平均絶対誤差（MAE）と二乗平均平方根誤差（RMSE）を関数にし、大きな誤差の扱いの違いを説明できる
- 線形回帰の係数を「ほかの列を同じにしたまま、その列を1増やすと、予測が何ドル変わるか」と読み、1列だけの式の係数と違う理由を説明できる
- 訓練とテストへの分け方を変えると、MAEも係数も揺れることを確かめ、似た2列が係数を不安定にすること（多重共線性）を説明できる
- 2つの式を比べるときに、同じ分け方どうしで並べる理由を説明できる
- 交差検証（cross-validation）で全部の行を1度ずつテストに回し、式を評価できる
- 関数を部品にして大きな関数を組み立て、1行を差し替えて別の評価の方法に変えられる
- f-stringで数を文の中に埋め込み、査定書の文を作れる


---


## 0. 準備


### 最初に実行する：道具を呼び出す

- 役割：このノートで使う道具（ライブラリ）を呼び出す
- 入力：なし
- 出力：なし（終わるとセルの左に ✓ が付く）

`import`は道具箱を持ってくる命令、`from`は道具箱から1つだけ取り出す書き方です。今日は、データの分け方を作る`KFold`が加わります（5節で使います）。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, KFold


### 図の日本語を表示できるようにする

- 役割：図の中の日本語が □ にならないようにし、図の大きさと字の大きさをそろえる
- 入力：なし
- 出力：「セットアップ完了」の表示

`try:`から先は、日本語表示の道具が動かなくなったときの保険で、中身は読まなくてかまいません。


In [ ]:
# 日本語が図の中で □ になるのを防ぐ
!pip install -q japanize-matplotlib
try:
    import japanize_matplotlib
# japanize が動かなくなったときの保険：同梱フォントを直接登録する
except Exception:
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = None
    if spec:
        ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None)
    if ttf:
        fm.fontManager.addfont(str(ttf))
        name = fm.FontProperties(fname=str(ttf)).get_name()
        plt.rcParams['font.family'] = name

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False  # マイナス記号の化けを防ぐ
print('セットアップ完了')


### お手本：表を読み込んで、列名を日本語にする

- 役割：CSVファイルの表を読み込み、今日使う列の名前を日本語に付け替えて、先頭5行を見る
- 入力：データの置き場所`DATA_DIR`と、ファイル名`ames_housing.csv`
- 出力：表`raw`（元の表）

第3回と同じく、`{'元の名前': '新しい名前', ...}`の辞書を`rename(columns=...)`に渡して、列名を付け替えます。80本のうち、今日使う13本だけを日本語にします。


In [ ]:
# データはこのリポジトリから読み込む
DATA_DIR = ('https://raw.githubusercontent.com/k0heiun0/'
            'applied_exercise_preview/main/shared/data')
raw = pd.read_csv(f'{DATA_DIR}/ames_housing.csv')
ja_names = {'Sale Condition': '売買の条件', 'SalePrice': '価格',
            'Overall Qual': '品質', 'Gr Liv Area': '居住面積',
            'Total Bsmt SF': '地下の面積',
            'Garage Cars': '車庫の台数', 'Garage Area': '車庫の面積',
            'Year Built': '建てた年', 'Year Remod/Add': '改装した年',
            'Full Bath': 'バスルーム数', 'TotRms AbvGrd': '部屋数',
            'Fireplaces': '暖炉の数', 'Lot Area': '敷地面積'}
raw = raw.rename(columns=ja_names)
print(len(raw), '件')
raw[['品質', '居住面積', '車庫の台数', '売買の条件', '価格']].head()


日本語にした列の中身は、次のとおりです。

| 日本語の名前 | 元の名前 | 中身 |
|---|---|---|
| `売買の条件` | `Sale Condition` | ふつうの売買（`Normal`）・新築の売り出し・親族間の売買など |
| `価格` | `SalePrice` | 売れた価格（米ドル）。当てたい値 |
| `品質` | `Overall Qual` | 家全体の材料と仕上げの品質（1〜10の10段階） |
| `居住面積` | `Gr Liv Area` | 地上階の居住面積 |
| `地下の面積` | `Total Bsmt SF` | 地下室の面積 |
| `車庫の台数`・`車庫の面積` | `Garage Cars`・`Garage Area` | 車庫に入る車の台数と、車庫の面積 |
| `建てた年`・`改装した年` | `Year Built`・`Year Remod/Add` | 建てた年と、改装した年（改装していなければ建てた年） |
| `バスルーム数` | `Full Bath` | 地上階の、浴槽かシャワーのあるバスルームの数 |
| `部屋数` | `TotRms AbvGrd` | 地上階の部屋の数（バスルームを除く） |
| `暖炉の数` | `Fireplaces` | 暖炉の数 |
| `敷地面積` | `Lot Area` | 敷地の面積 |

面積の列は、元は平方フィート（1平方フィートは約0.0929㎡）です。


### 変えてみる：ふつうの売買に絞り、面積を㎡に直す

- 役割：ふつうの売買で、居住面積が4,000平方フィート以下の家に絞り、面積の列を㎡に直す
- 入力：表`raw`
- 出力：絞った表`df`と、その軒数・価格の要約

親族間の売買や、差し押さえの売買は、ふつうの値段の付き方と違います。居住面積が4,000平方フィート（約372㎡）を超える家は全体で5軒だけで、データの作者も、授業で使うなら外すよう勧めています。

`&`は「かつ」です（第3回のおさらい）。`for`文で、面積の4本の列に同じ計算（0.0929を掛ける）を順に当てます。


In [ ]:
normal = raw['売買の条件'] == 'Normal'     # ふつうの売買
small = raw['居住面積'] <= 4000            # 広すぎない家
df = raw[normal & small].reset_index(drop=True)
for c in ['居住面積', '地下の面積', '車庫の面積', '敷地面積']:
    df[c] = df[c] * 0.0929                 # 平方フィート → ㎡
print(len(df), '軒')
print(df['価格'].describe().round(0))


残ったのは2,412軒です。価格の平均は約17万5千ドル、中央値は約15万9千ドルで、最も安い家は3万5千ドル、最も高い家は62万5千ドルです。


### 手がかりにする11本の列

価格を当てる手がかりにする列の名前を、`feat`にしまっておきます。今日は、この11本の列をいろいろに選び直して、式を比べます。


In [ ]:
feat = ['品質', '居住面積', '地下の面積', '車庫の台数', '車庫の面積',
        '建てた年', '改装した年', 'バスルーム数', '部屋数',
        '暖炉の数', '敷地面積']
print(len(feat), '本')


### 分析の地図：今日はここ

データ解析は「① データの理解と目標の設定 → ② 前処理（preprocessing）とデータ解析 → ③ 結果の解釈と目標との整合」の3つのフェーズを回ります。今日は、②の評価と、③の解釈が中心です。


In [ ]:
# @title 図：分析の地図
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis('off')
phases = ['① データの理解と\n目標の設定', '② 前処理と\nデータ解析', '③ 結果の解釈と\n目標との整合']
colors = ['#0066cc', '#2a9d8f', '#e63946']
for i, (x0, p, c) in enumerate(zip([0.17, 0.5, 0.83], phases, colors), start=1):
    on = i in (2, 3)
    ax.text(x0, 0.62, p, ha='center', va='center', fontsize=13 if on else 11, color='white',
            bbox=dict(boxstyle='round,pad=0.6', facecolor=c, alpha=0.95 if on else 0.3))
for a0, a1 in [(0.29, 0.365), (0.62, 0.695)]:
    ax.annotate('', xy=(a1, 0.62), xytext=(a0, 0.62),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()


### 今日の道筋

次の6つの小さな問いに答えていきます。

1. 何ドル外れる？（1節）
2. 係数は何を言っている？（2節）
3. その数字、どこまで信じる？（3節）
4. 列を1本外すと？（4節）
5. 全部の行を1度ずつテストにするには？（5節）
6. どちらを外し、どう説明する？（6〜7節）


---


## 1. 何ドル外れる？


表が整いました。まず第2回と同じ手順で、訓練とテストに分けて線形回帰の式を作り、テストの家で当たり具合を測ります。


### お手本：分けて、式を作る

- 役割：家を訓練8割・テスト2割に分け、11本の列で価格を当てる式を作る
- 入力：表`df`と、列の名前`feat`
- 出力：訓練とテストの軒数と、テストでの決定係数（R²）

`m.score(テストの手がかり, テストの答え)`は、第2回で使った決定係数（coefficient of determination）R²を返します。今日は、分け方の乱数（random number）の種を`random_state=0`にします。


In [ ]:
tr, te = train_test_split(df, test_size=0.2, random_state=0)
m = LinearRegression().fit(tr[feat], tr['価格'])
print(len(tr), len(te))
print(round(m.score(te[feat], te['価格']), 3))   # テストのR²


訓練1,929軒、テスト483軒で、テストでのR²は0.854です。価格のばらつきの8割以上を、式で説明できています。

R²は、どれだけ説明できたかの割合です。でも、査定担当として知りたいのは「何ドル外れるか」です。


### おさらい：MAEの関数

第3回で書いた、平均絶対誤差（MAE）の関数です。$y_i$は$i$番目の家の実際の価格、$\hat{y}_i$はその予測です。

$$
\text{MAE} = \frac{1}{n}\sum_{i=1}^{n} \lvert y_i - \hat{y}_i \rvert
$$

`def 名前(引数):`で関数を作り、`return`の後ろが戻り値です。作っただけでは何も出ないので、呼び出して使います。


In [ ]:
def mae(y, pred):          # MAE
    return round(np.abs(y - pred).mean())

pred = m.predict(te[feat])
print(mae(te['価格'], pred), 'ドル')


テストの家での予測は、平均19,391ドル外れます。価格の中央値が約15万9千ドルなので、1割ほどです。


### お手本：大きな誤差を重く見る物差し（RMSE）

MAEは、誤差を大きさのまま平均します。5千ドルの誤差が10軒あっても、5万ドルの誤差が1軒と9軒の正解でも、MAEは同じ5千ドルです。でも、査定では、5万ドル外した1軒が、お客さんの信頼を大きく失わせます。

そこで、誤差を二乗してから平均し、最後に平方根で単位をドルに戻す物差しを使います。二乗平均平方根誤差（RMSE, root mean squared error）と呼びます。

$$
\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}
$$

二乗すると、大きな誤差ほど、もっと大きくなります。RMSEは、大きな誤差を重く見る物差しです。

- 役割：実際の価格と予測から、RMSE（ドル）を出す
- 入力：実際の価格`y`と、予測`pred`
- 出力：RMSE（整数に丸めたドル）

`mae`との違いは1行目の中だけです。`** 2`は二乗、`np.sqrt`は平方根です。


In [ ]:
def rmse(y, pred):         # RMSE
    return round(np.sqrt(((y - pred) ** 2).mean()))

print(rmse(te['価格'], pred), 'ドル')


RMSEは27,191ドルで、MAE（19,391ドル）より大きくなりました。RMSEは、いつもMAE以上になります。差が大きいほど、少数の大きな誤差が混ざっています。


### TODO①：5万ドル以上外した家を数える

テストの家のうち、予測が5万ドル以上外れた家は何軒あるでしょうか。誤差の大きさ`np.abs(te['価格'] - pred)`が50000以上の家の数と、誤差のいちばん大きな値を表示してください。


In [ ]:
# TODO: 誤差の大きさが 50000 以上の家の数と、誤差の最大値を表示する
# ヒント: err = np.abs(te['価格'] - pred) のあと、
#         (err >= 50000).sum() と err.max()
...


483軒のうち28軒で、5万ドル以上外れています。いちばん大きな誤差は約20万ドル（200,677ドル）です。MAEだけを見ていると、こうした大きな誤差が見えません。このあとは、MAEとRMSEを並べて測ります。


---


## 2. 係数は何を言っている？


何ドル外れるかが分かりました。次は、お客さんへの説明です。この式は、価格について何を言っているのでしょうか。式の係数を読みます。


### 係数の読み方

線形回帰の式は、11本の列$x_{i1}, \dots, x_{i11}$に係数$b_1, \dots, b_{11}$を掛けて足し、切片$b_0$を加えたものです。

$$
\hat{y}_i = b_0 + b_1 x_{i1} + b_2 x_{i2} + \cdots + b_{11} x_{i11}
$$

ほかの列の値を全部そのままにして、$x_{ij}$だけを1増やすと、予測$\hat{y}_i$は$b_j$だけ変わります。係数$b_j$は「ほかの列を同じにしたまま、その列を1増やすと、予測が何ドル変わるか」です。


### お手本：係数を、列の名前と並べる

- 役割：1節の式`m`の係数を、列の名前と並べて表示する
- 入力：1節の式`m`
- 出力：列ごとの係数（ドル）

`m.coef_`には、係数が`feat`と同じ順に並んでいます。`pd.Series(値の並び, index=名前の並び)`で、名前の付いた1列の表にします。名前が付くと、`coef['品質']`のように名前で取り出せます。


In [ ]:
coef = pd.Series(m.coef_, index=feat)
print(coef.round(0))


例えば品質は16,065ドルです。「ほかの条件が同じなら、品質が1段階上がると、予測が約1万6千ドル上がる」と読みます。居住面積は675ドルで、「ほかが同じなら、居住面積が1㎡広いと675ドル高い」です。暖炉の数は8,548ドルです。

一方で、気になる係数が2つあります。車庫の台数は267ドルです。車庫に車がもう1台入っても、たった267ドルしか上がらないのでしょうか。そして、バスルーム数は−9,976ドルです。バスルームが1つ多いと、約1万ドル安くなるのでしょうか。


### 変えてみる：バスルーム数だけの式

- 役割：バスルーム数だけで価格を当てる式の係数と、バスルーム数・居住面積・部屋数の相関を見る
- 入力：訓練の表`tr`
- 出力：1列だけの式の係数と、3つの列の相関係数の表

`corr()`は、列どうしの相関係数を表にします。


In [ ]:
m_bath = LinearRegression().fit(tr[['バスルーム数']], tr['価格'])
print(round(m_bath.coef_[0]), 'ドル')
print(tr[['バスルーム数', '居住面積', '部屋数']].corr().round(2))


バスルーム数だけの式では、係数は68,759ドルです。バスルームが多い家は、高いのです。

バスルーム数と居住面積の相関は0.64、部屋数との相関は0.53です。バスルームの多い家は、たいてい広くて部屋も多い家です。1列だけの式の68,759ドルには、「広くて部屋も多い」ことの分まで入っています。

11本の式の−9,976ドルは、意味が違います。居住面積と部屋数を同じにしたまま、バスルームだけを1つ増やすと、という比較です。同じ広さでバスルームが1つ多いなら、そのぶんほかの部屋が狭くなります。「バスルームが多い家は安い」ではなく、「同じ広さなら、バスルームに面積を割いた家は、少し安い」です。

係数は、一緒に入れた列によって意味が変わります。いつも「ほかの列を同じにしたら」を付けて読みます。


---


## 3. その数字、どこまで信じる？


係数が読めるようになりました。ただ、1節のMAEも2節の係数も、`random_state=0`という1通りの分け方で出した数です。分け方を変えたら、同じ数が出るのでしょうか。


### おさらい：for文

`for 変数 in 並び:`は、並びの中身を1つずつ変数に入れて、字下げした行をくり返します。`range(20)`は0から19までの20個の数です。

```python
for s in range(3):
    print(s)          # 0, 1, 2 と順に表示される
```


### お手本：分け方を20通り変える

- 役割：`random_state`を0〜19に変えて分け直し、毎回、式を作ってMAEと車庫の台数の係数をためる
- 入力：表`df`と、列の名前`feat`
- 出力：20回分の結果の表`result`の要約、MAEの最大と最小の差、係数がマイナスになった回数

`rows = []`で空のリストを作り、毎回の結果を辞書`{'名前': 値, ...}`にして`rows.append(...)`で1行ずつ足します。最後に`pd.DataFrame(rows)`で、20行の表にします。

ループの中の訓練とテストは、1節の`tr`・`te`を上書きしないよう、`tr_s`・`te_s`という名前にします。


In [ ]:
rows = []
for s in range(20):
    tr_s, te_s = train_test_split(df, test_size=0.2, random_state=s)
    m_s = LinearRegression().fit(tr_s[feat], tr_s['価格'])
    coef_s = pd.Series(m_s.coef_, index=feat)
    rows.append({'MAE': mae(te_s['価格'], m_s.predict(te_s[feat])),
                 '車庫の台数': round(coef_s['車庫の台数'])})
result = pd.DataFrame(rows)
print(result.describe().round(0))
print(result['MAE'].max() - result['MAE'].min(), 'ドルの幅')
print((result['車庫の台数'] < 0).sum(), '回はマイナス')


MAEは、最小18,555ドルから最大20,874ドルまで、分け方だけで2,319ドルも揺れます。平均は19,367ドルです。1節の19,391ドルは、この20通りのうちの1つ（`random_state=0`）にすぎません。どの分け方を引くかで、報告する数が変わってしまいます。


In [ ]:
plt.plot(result['MAE'], 'o')
plt.xlabel('random_state')
plt.ylabel('テストでのMAE（ドル）')
plt.show()


### 係数は、符号まで揺れる

車庫の台数の係数は、もっと揺れます。最小−2,081ドルから最大2,143ドルまで、20回のうち8回はマイナスです。分け方しだいで、「車庫に車がもう1台入ると、高くなる」とも「安くなる」とも読めてしまいます。これでは、お客さんに説明できません。


### お手本：車庫の台数と車庫の面積の相関

- 役割：車庫の台数と車庫の面積の相関係数を出す
- 入力：表`df`
- 出力：相関係数


In [ ]:
print(round(df['車庫の台数'].corr(df['車庫の面積']), 2))


相関係数は0.89です。車が何台入るかは、ほぼ車庫の広さで決まります。2つの列が、ほとんど同じことを言っているのです。

こうした似た2列を一緒に式に入れると、2つの係数の「和」はデータで決まっても、どちらにどれだけ割り振るかの「内訳」が決まりません。台数に多く割り振っても、面積に多く割り振っても、予測はほとんど変わらないからです。訓練に入る家が少し入れ替わるだけで、内訳が大きく動き、係数の符号まで変わります。

似た列が式の中にあって係数が不安定になることを、多重共線性（multicollinearity）と呼びます。


---


## 4. 列を1本外すと？


車庫の台数の係数が揺れるのは、よく似た車庫の面積が一緒に入っているからでした。それなら、車庫の面積を外せば、台数の係数は落ち着くはずです。ただ、手がかりを1本減らすと、当たりは悪くならないでしょうか。3節と同じ20回の評価を、列を変えてもう1度やります。


### お手本：20回の評価を関数にする

3節のセルを、列の組を変えてもう1度書くのは手間です。くり返しになったところで、関数にまとめます。

- 役割：分け方を20通り変えて式を作り、毎回のMAE・RMSE・指定した列の係数をためた表を返す
- 入力：手がかりの列の名前`cols`、係数を見る列`watch`（既定値は`'車庫の台数'`）
- 出力：20行の表（列はMAE・RMSE・係数）

中身は3節のセルとほぼ同じです。違いは3つです。`feat`を引数`cols`に変えたこと、1節の`rmse`も呼ぶようにしたこと、係数を見る列を引数`watch`で選べるようにしたことです。1節で作った`mae`と`rmse`を、この関数の中で部品として呼んでいます。


In [ ]:
def split_scores(cols, watch='車庫の台数'):
    rows = []
    for s in range(20):
        tr_s, te_s = train_test_split(df, test_size=0.2,
                                      random_state=s)
        m_s = LinearRegression().fit(tr_s[cols], tr_s['価格'])
        pred_s = m_s.predict(te_s[cols])
        coef_s = pd.Series(m_s.coef_, index=cols)
        rows.append({'MAE': mae(te_s['価格'], pred_s),
                     'RMSE': rmse(te_s['価格'], pred_s),
                     '係数': round(coef_s[watch])})
    return pd.DataFrame(rows)

all11 = split_scores(feat)
print(all11.mean().round(0))


MAEの平均は19,367ドルで、3節と同じです。同じ20通りの分け方で、同じ式を作っているからです。関数にまとめても、結果は変わりません。


### 変えてみる：車庫の面積を外す

- 役割：車庫の面積を外した10本の列で、同じ20回の評価をする
- 入力：列の名前`no_area`
- 出力：20回の結果の要約と、11本の式とのMAEの平均の差


In [ ]:
no_area = ['品質', '居住面積', '地下の面積', '車庫の台数',
           '建てた年', '改装した年', 'バスルーム数', '部屋数',
           '暖炉の数', '敷地面積']
drop_area = split_scores(no_area)
print(drop_area.describe().round(0))
gap = drop_area['MAE'].mean() - all11['MAE'].mean()
print(round(gap), 'ドル悪い')


車庫の台数の係数は、最小7,392ドルから最大9,801ドルで、20回とも約8千〜1万ドルのプラスに落ち着きました。「車庫に車がもう1台入ると、約8千ドル高い」と説明できます。

MAEの平均は19,651ドルで、11本の式（19,367ドル）より285ドル悪くなりました。


### 問い：285ドルの差は、揺れの中に埋もれていないか

MAEは、分け方だけで2,319ドルも揺れました。それに比べて、285ドルの差はずっと小さな差です。面積を外した式が本当に悪いのか、たまたまなのか、どうやって確かめればよいでしょうか。予想を下に書いてください。


（ここに予想を書く）


### お手本：同じ分け方どうしで並べる

- 役割：20回それぞれで、11本の式と面積を外した式のMAEを横に並べ、差を出す
- 入力：`split_scores`の2つの結果`all11`と`drop_area`
- 出力：20行の比較の表と、面積を外した式が悪かった回数

`random_state`が同じ回どうしは、テストの家がまったく同じです。同じ家で測った2つのMAEなら、差は式の違いだけから生まれます。


In [ ]:
compare = pd.DataFrame({'11本': all11['MAE'],
                        '面積を外す': drop_area['MAE']})
compare['差'] = compare['面積を外す'] - compare['11本']
print(compare)
print((compare['差'] > 0).sum(), '回は面積を外すと悪い')


20回すべてで、面積を外した式のほうが悪くなりました。差は、最も小さい回で50ドル、最も大きい回で547ドルです。MAEそのものは分け方で大きく揺れますが、同じ分け方どうしの差は、いつも同じ向きです。

逆に、違う分け方どうしで比べると、結論が逆になります。例えば、面積を外した式の`random_state=16`（18,730ドル）と、11本の式の`random_state=7`（20,874ドル）を比べると、面積を外した式のほうが2千ドル以上も良く見えます。テストの家が違えば、当てやすい家に当たったかどうかで、MAEが変わるからです。式を比べるときは、同じ分け方どうしで並べます。


In [ ]:
plt.plot(compare['11本'], 'o-', label='11本')
plt.plot(compare['面積を外す'], 's-', label='面積を外す')
plt.xlabel('random_state')
plt.ylabel('テストでのMAE（ドル）')
plt.legend()
plt.show()


2本の線は、分け方ごとに一緒に上がったり下がったりします。そして、どの分け方でも、面積を外した式（四角）が少し上にあります。

これで、板ばさみがはっきりしました。面積を外すと、係数は説明に使えるのに、当たりは確かに少し悪くなります。


---


## 5. 全部の行を1度ずつテストに


同じ分け方どうしで並べれば、式を比べられることが分かりました。ただ、20回の分け直しには、まだ気になる点があります。それぞれの家は、20回のうち何回テストに入ったのでしょうか。


### お手本：家ごとに、テストに入った回数を数える

- 役割：20回の分け方それぞれで、テストに入った家に1を足していき、家ごとの回数を数える
- 入力：表`df`
- 出力：「テストに入った回数」ごとの家の数

`pd.Series(0, index=df.index)`で、家の数だけ0を並べた入れ物を作ります。`count[te_s.index] += 1`は、その回にテストに入った家（`te_s.index`の行）にだけ1を足します。`value_counts()`は値ごとの数、`sort_index()`はそれを回数の小さい順に並べます。


In [ ]:
count = pd.Series(0, index=df.index)
for s in range(20):
    tr_s, te_s = train_test_split(df, test_size=0.2, random_state=s)
    count[te_s.index] += 1
print(count.value_counts().sort_index())


20回で1度もテストに入らなかった家が29軒あります。一方で、10回もテストに入った家が3軒あります。20回のMAEの平均は、何度もテストに入った家ほど重く数え、1度も入らなかった家は数えていないことになります。

どの家も、ちょうど1度ずつテストに入るように分けられないでしょうか。


### 交差検証：全部の行を1度ずつテストにする

家を、重ならない5つの組に分けます。1回目は1つ目の組をテストに、残りの4組を訓練にして、式を作ってMAEを測ります。2回目は2つ目の組をテストに……と、5回くり返します。5回で、どの家もちょうど1度ずつテストに入ります。最後に、5回のMAEを平均します。

$$
\text{CV} = \frac{1}{K}\sum_{k=1}^{K} \text{MAE}_k
$$

$K$は組の数（今日は5）、$\text{MAE}_k$は$k$番目の組をテストにしたときのMAEです。このやり方を、$K$分割交差検証（K-fold cross-validation, CV）と呼びます。


In [ ]:
# @title 図：5分割の交差検証
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.axis('off')
for k in range(5):
    for g in range(5):
        is_test = g == k
        ax.add_patch(plt.Rectangle((g * 1.0, 4 - k), 0.95, 0.8,
                                   color='#e63946' if is_test else '#0066cc',
                                   alpha=0.85 if is_test else 0.35))
    ax.text(-0.15, 4 - k + 0.4, f'{k + 1}回目', ha='right', va='center')
ax.text(5.1, 4.4, '赤：テスト\n青：訓練', va='center')
ax.set_xlim(-1.2, 6.5); ax.set_ylim(-0.2, 5)
plt.show()


### お手本：5つの組を作る

- 役割：家を5つの組に分ける道具を作り、5回分の訓練とテストの行番号を確かめる
- 入力：表`df`
- 出力：各回の訓練とテストの軒数と、テストに入った家の最初の5つの行番号

`KFold(n_splits=5, shuffle=True, random_state=0)`は、行を混ぜてから5つの組に分ける道具です。`kf.split(df)`は、5回分の「訓練の行番号の並び」と「テストの行番号の並び」の組を、`for`文で1回ずつ渡します。


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=0)
for tr_i, te_i in kf.split(df):
    print(len(tr_i), len(te_i), te_i[:5])


5回とも、訓練が1,929軒か1,930軒、テストが483軒か482軒です。テストに入る家は、回ごとに入れ替わります。

`tr_i`と`te_i`は行番号の並びです。表から、その行番号の行を取り出すには`df.iloc[行番号の並び]`を使います。`iloc`は、行の番号（0から数えた位置）で行を取り出す書き方です。


### お手本：交差検証の関数

- 役割：5分割の交差検証で、毎回のMAE・RMSE・指定した列の係数をためた表を返す
- 入力：手がかりの列の名前`cols`、係数を見る列`watch`（既定値は`'車庫の台数'`）
- 出力：5行の表（列はMAE・RMSE・係数）

4節の`split_scores`と比べてください。変わったのは、`for`の行と、その次の訓練とテストを作る行の2行だけです。「分け方」の部品を差し替えるだけで、20回の分け直しが交差検証に変わります。


In [ ]:
def cv_score(cols, watch='車庫の台数'):
    rows = []
    for tr_i, te_i in kf.split(df):
        tr_s, te_s = df.iloc[tr_i], df.iloc[te_i]
        m_s = LinearRegression().fit(tr_s[cols], tr_s['価格'])
        pred_s = m_s.predict(te_s[cols])
        coef_s = pd.Series(m_s.coef_, index=cols)
        rows.append({'MAE': mae(te_s['価格'], pred_s),
                     'RMSE': rmse(te_s['価格'], pred_s),
                     '係数': round(coef_s[watch])})
    return pd.DataFrame(rows)

cv11 = cv_score(feat)
print(cv11)
print(cv11.mean().round(0))


5回のMAEは18,862〜19,459ドルで、平均は19,297ドルです。RMSEの平均は26,997ドルです。1回目は19,391ドルで、1節と同じ数です。`KFold`と`train_test_split`に同じ`random_state=0`を渡したので、1回目のテストの組が、1節のテストとたまたま同じ483軒になりました。

車庫の台数の係数は、交差検証でも267ドルから−1,032ドルまで揺れ、5回のうち4回はマイナスです。


### TODO②：面積を外した式も、交差検証で比べる

`cv_score`に、4節の`no_area`を渡して、面積を外した式の5回分の結果を出してください。そのうえで、11本の式（`cv11`）とMAEを横に並べた表を作り、5回のうち何回、面積を外した式のほうが悪かったかを表示してください。


In [ ]:
# TODO: cv_score(no_area) の結果と cv11 の MAE を並べて比べる
# ヒント: cv_area = cv_score(no_area) のあと、4節の compare と同じ形で
#         cv11['MAE'] と cv_area['MAE'] を並べた表を作る
...


面積を外した式のMAEは平均19,561ドルで、5回すべてで11本の式より悪くなります。交差検証でも、4節と同じ結論です。

交差検証は、どの家もちょうど1度ずつテストに入るので、家の偏りなく式を評価できます。そして、2つの式を同じ5つの組で評価するので、いつも同じ分け方どうしで並べて比べられます。式を比べるときの、標準の物差しです。


---


## 6. どちらを外す？


交差検証で確かめても、車庫の面積を外すと当たりは悪くなりました。でも、似た2列のうち、外せるのは面積だけではありません。台数のほうを外したら、どうなるでしょうか。


### TODO③：車庫の台数を外して、交差検証で評価する

車庫の台数を外した10本の列の名前`no_cars`を作り、`cv_score`で評価してください。台数を外したので、係数は車庫の面積の係数を見ます（`watch='車庫の面積'`）。5回分の表と、その平均を表示してください。


In [ ]:
# TODO: 車庫の台数を外した列の組 no_cars を作り、cv_score で評価する
# ヒント: no_cars は feat から '車庫の台数' を除いた10本。
#         cv_score(no_cars, watch='車庫の面積')
...


### お手本：3つの式を、1つの表に並べる

- 役割：11本の式・面積を外した式・台数を外した式を、同じ交差検証で評価し、平均を1つの表に並べる
- 入力：3つの列の組
- 出力：式ごとのMAE・RMSE・係数の平均

`cv_score(...).mean()`は、5回分の表の列ごとの平均です。それを辞書の値にして`pd.DataFrame`に渡すと、式ごとに1列の表になります。係数の行は、11本と面積を外した式では車庫の台数、台数を外した式では車庫の面積の係数です。


In [ ]:
no_cars = ['品質', '居住面積', '地下の面積', '車庫の面積',
           '建てた年', '改装した年', 'バスルーム数', '部屋数',
           '暖炉の数', '敷地面積']
table = pd.DataFrame({
    '11本': cv_score(feat).mean(),
    '面積を外す': cv_score(no_area).mean(),
    '台数を外す': cv_score(no_cars, watch='車庫の面積').mean()})
print(table.round(0))


台数を外した式のMAEは19,297ドルで、11本の式と同じです。RMSEも26,994ドルで、11本の式（26,997ドル）とほとんど変わりません。車庫の台数を外しても、当たりは落ちないのです。

車庫の面積の係数は、平均で1㎡あたり390ドルです。


In [ ]:
cv_cars = cv_score(no_cars, watch='車庫の面積')
print(cv_cars)


5回とも、車庫の面積の係数は1㎡あたり375〜402ドルのプラスで、ほとんど揺れません。MAEも、5回それぞれで11本の式とほぼ同じです（1回目は19,386ドル、11本は19,391ドル）。

なぜ、台数を外しても当たりが落ちないのでしょうか。車庫の台数ごとに、車庫の面積の平均を見ます。


In [ ]:
print(df.groupby('車庫の台数')['車庫の面積'].mean().round(0))


車1台の車庫は平均28㎡、2台は48㎡、3台は74㎡です。台数が1台増えると、面積は20〜26㎡ほど増えます。車庫の面積が、台数の情報をほとんど持っているのです。1㎡あたり390ドルなら、20〜26㎡で約8千〜1万ドルで、4節で面積を外したときの台数の係数（約8千〜1万ドル）とほぼ同じ大きさです。

反対に、台数は面積の情報を全部は持っていません。同じ2台用でも、広い車庫と狭い車庫があるからです。だから、面積を外すと当たりが少し落ち、台数を外しても落ちませんでした。

似た2列のどちらを残すかは、交差検証で比べて決めます。今日の査定には、車庫の台数を外した10本の式を使います。当たりは11本の式と同じで、係数は揺れずに読めます。


---


## 7. この家は、なぜこの値段？


査定に使う式が決まりました。いよいよ、1軒の家の査定書を書きます。


### お手本：選んだ式を、全部の家で作り直す

- 役割：台数を外した10本の列で、2,412軒すべてを使って式を作り直し、係数を表示する
- 入力：表`df`と、列の名前`no_cars`
- 出力：列ごとの係数

交差検証は、式を選ぶための評価でした。選んだら、手元の家を全部使って、査定に使う式を作り直します。家が多いほど、係数は安定するからです。当たりの目安には、交差検証のMAE（19,297ドル）を使います。


In [ ]:
final = LinearRegression().fit(df[no_cars], df['価格'])
coef_f = pd.Series(final.coef_, index=no_cars)
print(coef_f.round(0))


品質は1段階で15,803ドル、居住面積は1㎡で686ドル、車庫の面積は1㎡で391ドルです。バスルーム数は−10,245ドルで、2節と同じく「同じ広さなら、バスルームに面積を割いた家は少し安い」です。


### 予測を、条件ごとの金額に分ける

査定書では、「平均的な家より、なぜ高い（安い）のか」を説明します。

式の予測$\hat{y}$から、平均的な家（どの列も2,412軒の平均$\bar{x}_j$）の予測を引くと、切片が消えて、列ごとの金額の和になります。

$$
\hat{y} - \hat{y}_{\text{平均的な家}} = \sum_j b_j (x_j - \bar{x}_j)
$$

$b_j(x_j - \bar{x}_j)$が、列$j$がこの家を平均よりいくら押し上げたか（マイナスなら押し下げたか）です。


### お手本：1軒目の家の内訳

- 役割：1軒目の家の予測と、平均的な家の予測、列ごとの押し上げを出す
- 入力：選んだ式`final`と係数`coef_f`、1軒目の家
- 出力：予測・平均的な家の予測・その差、家の条件の一部、列ごとの押し上げ

`df.iloc[[0]]`は、0行目の家を1行の表のまま取り出します（かっこを二重にすると表のまま）。`house.iloc[0]`は、その1行を列の名前の付いた並びにします。係数と同じく列の名前が付いているので、`coef_f * (家の値 - 平均)`は列どうしで掛け算されます。


In [ ]:
house = df.iloc[[0]]
pred_h = final.predict(house[no_cars])[0]
avg_h = final.predict(df[no_cars]).mean()   # 平均的な家の予測
push = coef_f * (house[no_cars].iloc[0] - df[no_cars].mean())
print(round(pred_h), round(avg_h), round(pred_h - avg_h))
print(house[['敷地面積', '暖炉の数', '建てた年', '改装した年']])
print(push.round(0))


1軒目の家の予測は211,548ドル、平均的な家の予測は175,327ドルです。差の36,221ドルを、列ごとの押し上げに分けました。

敷地面積が16,815ドル、居住面積が11,463ドル、暖炉の数が11,193ドルの押し上げです。この家は、敷地が約2,951㎡と広く、暖炉が2つあります。一方で、改装した年は−8,195ドルの押し下げです。1960年に建てたまま、改装していないからです。


### 問い：バスルームが少ないのに、押し上げ？

バスルーム数の押し上げは、+5,518ドルです。この家のバスルームは1つで、平均より少ないのに、値段を押し上げています。お客さんに「バスルームが少ないので高くなりました」と書いてよいでしょうか。予想を下に書いてください。


（ここに予想を書く）


そう書くと、お客さんは混乱します。バスルーム数の係数は「居住面積と部屋数を同じにしたら」の値でした。この家は、居住面積が平均より広いのに、バスルームが1つしかありません。そのぶん、ほかの部屋に面積を回している、という意味です。

バスルーム数は、居住面積・部屋数と似た列で、3つで「広さと間取り」を表しています。こうした列は、1つずつ説明せず、合わせて読みます。居住面積（+11,463ドル）・バスルーム数（+5,518ドル）・部屋数（−2,078ドル）を足すと、「広さと間取り」で+14,903ドルです。


### お手本：査定書の文を作る

- 役割：予測・交差検証のMAE・押し上げを、査定書の文にして表示する
- 入力：1軒目の家の予測`pred_h`、平均的な家の予測`avg_h`、押し上げ`push`
- 出力：査定書の文（5行）

先に、文に入れる金額を短い名前の変数にしておきます（`space`・`land`など）。

文字の前に`f`を付けた`f'...'`（f-string）の中では、`{ }`で囲んだところに、変数や計算の値が入ります。`{ }`の中の`:`の後ろは、数の書き方の指定です。

- `{pred_h:,.0f}`：3けたごとにカンマを入れ（`,`）、小数点以下0けた（`.0f`）で書く
- `{space:+,.0f}`：さらに、プラスのときも`+`を付ける（`+`）


In [ ]:
space = push['居住面積'] + push['バスルーム数'] + push['部屋数']
land = push['敷地面積']
fire = push['暖炉の数']
remod = push['改装した年']
cv_err = round(cv_score(no_cars, watch='車庫の面積')['MAE'].mean())
print(f'査定額：{pred_h:,.0f}ドル')
print(f'似た家での誤差は、平均{cv_err:,}ドルほどです。')
print(f'平均的な家（{avg_h:,.0f}ドル）より{pred_h - avg_h:+,.0f}ドル')
print(f'広さと間取り{space:+,.0f}ドル、敷地{land:+,.0f}ドル')
print(f'暖炉{fire:+,.0f}ドル、改装していないこと{remod:+,.0f}ドル')


査定書の文ができました。数を手で書き写さずに、計算した値をそのまま文に入れるので、式を作り直しても文がずれません。誤差の目安`cv_err`も、6節の交差検証をもう1度呼んで出しています。


### TODO④：別の家の査定書

1軒目の代わりに、`df.iloc[[5]]`（6軒目の家）の査定書を作ってください。お手本の「1軒目の家の内訳」と「査定書の文」の2つのセルを写し、`house = df.iloc[[0]]`の`0`を`5`に変えるだけで動きます。押し上げの大きい列を見て、査定書の最後の2行に書く列を選び直してみましょう。


In [ ]:
# TODO: 6軒目の家（df.iloc[[5]]）の押し上げと査定書の文を表示する
# ヒント: house = df.iloc[[5]] にして、お手本の2つのセルを写す。
#         print(push.round(0)) で押し上げの大きい列を確かめる
...


6軒目の家の予測は190,365ドルで、実際の価格は195,500ドルでした。平均的な家より15,037ドル高く、いちばん押し上げているのは建てた年（+8,485ドル）と居住面積（+8,147ドル）です。


### 答え

査定には、車庫の台数を外した10本の列の線形回帰を使います。5分割の交差検証でMAEは19,297ドルで、11本の式と同じ当たりです。そのうえで、車庫の面積の係数は1㎡あたり375〜402ドルと揺れず、お客さんに「ほかの条件が同じなら」と説明できます。

1軒目の家の査定額は211,548ドルで、平均的な家より36,221ドル高い値段です。広さと間取り・広い敷地・2つの暖炉が押し上げ、改装していないことが押し下げています。


まだ言えないこと

- 列を外さずに係数を落ち着かせる方法：似た列を外すと、その列の情報（同じ2台用でも広い車庫と狭い車庫の違いなど）は捨てることになる
- 誤差の幅は家ごとに違う：交差検証のMAEは家全体の平均で、高い家ほど大きく外れやすい
- 列の選び方：今日は11本から1本ずつ外して比べた。手がかりになる列はほかにも70本近くある


---


## 目標に答えられたか


- 今日の目標は「お客さんに説明できる査定の式を、どう選ぶか」でした。選んだ式と、選んだ理由を2つの面から説明できますか？
- 1節で、MAEとRMSEはどう違いましたか？5万ドル以上外した家は何軒ありましたか？
- 2節で、バスルーム数の係数が、1列だけの式と11本の式で逆の符号になったのはなぜですか？
- 3節で、分け方を変えると、MAEと車庫の台数の係数はどれくらい揺れましたか？ 係数が揺れる理由は何でしたか？
- 4節で、2つの式を比べるときに、同じ分け方どうしで並べたのはなぜですか？
- 5節の交差検証は、20回の分け直しと比べて、何が良いのでしょうか？
- 7節の査定書で、バスルーム数をほかの列と合わせて読んだのはなぜですか？


---


## 今日の要点


- 今日、持ち帰る1文：式は2つの面から選ぶ。1つは、交差検証で同じ分け方どうしを並べた当たりの良さ。もう1つは、係数が「ほかの列を同じにしたら」の値として、揺れずに読めるかどうか
- MAEは誤差の大きさの平均、RMSEは大きな誤差を重く見る物差し。RMSEはいつもMAE以上
- 係数は「ほかの列を同じにしたまま、その列を1増やすと、予測が何ドル変わるか」。一緒に入れた列によって意味が変わる
- 1回の分け方の数は揺れる。MAEは分け方だけで2千ドル以上動き、似た列の係数は符号まで変わる（多重共線性）
- 2つの式は、同じ分け方どうしで並べて比べる。違う分け方どうしでは、結論が逆にもなる
- 交差検証は、全部の行を1度ずつテストに回す。式を比べるときの標準の物差し
- 似た2列のどちらを残すかは、交差検証で比べて決める。似た列どうしは、合わせて読む
- Python：小さな関数（`mae`・`rmse`）を部品にして大きな関数（`split_scores`・`cv_score`）を組み立て、「分け方」の行を差し替えて評価の方法を変える。f-stringで数を文に埋め込む


---


## 次回へ


今日は、似た2列の片方を外して、係数を説明に使える形にしました。ただ、外した列の情報は捨てています。手がかりの列を増やすほど、似た列も増えていきます。

次回の問いは、列を増やして当たりを良くしながら、お客さんに説明できる係数を保てるか、です。


---


## 課題（提出）


提出するもの：応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

7節の査定書では、居住面積・バスルーム数・部屋数を「広さと間取り」として合わせて読みました。部屋数の係数も、査定に使った式ではマイナスです。では、部屋数を外したらどうなるでしょうか。

6節で選んだ10本の列`no_cars`から、さらに部屋数を外した9本の列で、`cv_score`による交差検証をしてください。係数は居住面積の係数を見ます（`watch='居住面積'`）。あわせて、部屋数だけで価格を当てる式の係数も表示してください。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

9本の列は、`no_cars`から`'部屋数'`を除いたリストを書きます。
`cv_score(9本の列, watch='居住面積')`で5回分の表が出ます。`.mean()`で平均を出し、6節の`table`の「台数を外す」の列と比べましょう。
部屋数だけの式は、2節の`m_bath`と同じ形で、`df[['部屋数']]`を使います。

</details>


### 応用②（判断）

応用①で、部屋数を外した9本の式の、交差検証のMAEの平均は何ドルですか。整数に丸めて答えてください（例: 12345）。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

`round(値)`で整数に丸めます。10本の式（19,297ドル）とも比べておきましょう。

</details>


### 応用③（解釈）

応用①の結果をもとに、「査定に使う式に、部屋数を入れておくか」を判断し、その理由を、お客さんへの説明の面と当たりの面から、3行程度で書いてください。


（ここに3行程度で書く）


---


## 発展（任意）


### 価格を対数にした式：係数を「約何%」と読む

第3回で、右に尾の長い料金を対数にすると、式で表しやすくなることを見ました。住宅の価格も、右に尾が長い量です。価格を対数にして当て、`np.exp`で元に戻す式を、交差検証で評価します。

- 役割：`cv_score`の中で、当てる値を$\log y$にし、予測を`np.exp`でドルに戻して、MAEを測る
- 入力：手がかりの列の名前`cols`
- 出力：5回のMAEの表

`cv_score`と比べると、変わったのは`fit`の行（当てる値を`np.log`に）と、予測の行（`np.exp`で戻す）の2行です。


In [ ]:
def cv_log(cols):
    rows = []
    for tr_i, te_i in kf.split(df):
        tr_s, te_s = df.iloc[tr_i], df.iloc[te_i]
        m_s = LinearRegression().fit(tr_s[cols],
                                     np.log(tr_s['価格']))
        pred_s = np.exp(m_s.predict(te_s[cols]))
        rows.append({'MAE': mae(te_s['価格'], pred_s),
                     'RMSE': rmse(te_s['価格'], pred_s)})
    return pd.DataFrame(rows)

print(cv_log(no_cars).mean().round(0))


MAEは16,051ドルで、対数にしない式（19,297ドル）より3千ドル以上小さくなります。RMSEも22,533ドルまで下がります。当たりは、対数の式のほうがはっきり良いのです。

ただ、係数の読み方が変わります。$\log \hat{y}$が$b_j$だけ増えると、$\hat{y}$は$e^{b_j}$倍になります。$b_j$が小さいときは$e^{b_j} \approx 1 + b_j$なので、係数を100倍した数が「約何%高くなるか」です。


In [ ]:
m_log = LinearRegression().fit(df[no_cars], np.log(df['価格']))
coef_log = pd.Series(m_log.coef_, index=no_cars)
print((coef_log * 100).round(2))                  # 約何%
print(((np.exp(coef_log) - 1) * 100).round(2))   # 正確に何%


品質の係数は0.0822で、「ほかの条件が同じなら、品質が1段階上がると、約8.2%高い」と読みます（正確には8.57%）。居住面積は1㎡で約0.30%、車庫の面積は1㎡で約0.23%です。

「何ドル高い」ではなく「何%高い」なので、安い家と高い家で、同じ条件の差が違う金額になります。品質が1段階上がると、30万ドルの家は10万ドルの家の3倍の金額だけ高くなります。住宅の価格は、こちらのほうが実感に合いそうです。お客さんに「何%」で説明できるなら、対数の式を選ぶのも1つの判断です。


### 自分の関数でVIFを出す

3節で、似た2列は係数を不安定にする（多重共線性）ことを見ました。どの列が、ほかの列とどれだけ似ているかを、1つの数で表せます。

列$j$を、ほかの全部の列で当てる線形回帰の式を作り、その決定係数を$R_j^2$とします。$R_j^2$が1に近いほど、列$j$はほかの列で説明できてしまう（似た列がある）ということです。

$$
\text{VIF}_j = \frac{1}{1 - R_j^2}
$$

この数を、分散拡大係数（VIF, variance inflation factor）と呼びます。列$j$の係数の揺れ（分散）が、似た列のせいで何倍に膨らむかを表します。似た列がなければ1で、大きいほど係数が揺れやすくなります。

- 役割：列の組`cols`の各列について、ほかの列で当てたR²からVIFを出す
- 入力：列の名前`cols`
- 出力：列ごとのVIF

`others`は、`cols`から列`c`を除いた列の名前のリストです。`for`文で`cols`を1つずつ見て、`c`でない列だけを`append`で集めます。


In [ ]:
def vif(cols):
    out = {}
    for c in cols:
        others = []
        for o in cols:
            if o != c:
                others.append(o)
        m_v = LinearRegression().fit(df[others], df[c])
        r2 = m_v.score(df[others], df[c])
        out[c] = round(1 / (1 - r2), 2)
    return pd.Series(out)

print(pd.DataFrame({'11本': vif(feat), '台数を外す': vif(no_cars)}))


11本の列では、車庫の台数のVIFが5.42、車庫の面積が4.95で、いちばん大きな組です。居住面積も4.82と大きめです（部屋数・バスルーム数と似ているため）。

車庫の台数を外すと、車庫の面積のVIFは1.65まで下がります。似た相手がいなくなったので、係数が揺れにくくなったのです。6節で、台数を外すと車庫の面積の係数が5回とも375〜402ドルに収まったことと合います。

VIFは、どの列が似た列を持っているかを見つける目安です。ただ、どちらの列を外すか、あるいは外さないかは、VIFだけでは決まりません。今日のように、交差検証での当たりと、係数を説明に使えるかの両方で決めます。

試すなら、4節の`no_area`（車庫の面積を外した10本）でもVIFを出し、車庫の台数のVIFがどこまで下がるかを確かめてみましょう。
